Let $\vec{g} := (g_1,\dots,g_k)$ and $\vec{h} := (h_1,\dots, h_r)$. Then,
$$\textbf{num\_forb\_subs}(G, \vec{g}, \vec{h}) := \# \{ A \subset G \mid \exists g_i \not \in AA \text{ or } \exists h_j \not \in AA^{-1}   \}$$
<!-- $$ = \# \{ A \subset G \mid g_1 \not \in AA \lor \cdots \lor g_k \not \in AA \lor h_1 \not \in AA^{-1} \lor \cdots \lor h_r \not \in AA^{-1} \}$$
$$ = \sum_{A \subset G}\mathbb{1}_{AA}(g_1)\cdots\mathbb{1}_{AA}(g_k)\cdot \mathbb{1}_{AA^{-1}}(h_1) \cdots \mathbb{1}_{AA^{-1}}(h_r) $$ -->

In [1]:
from itertools import combinations
from tqdm.notebook import tqdm
import random
from collections import Counter
from sage.graphs.independent_sets import IndependentSets
from math import comb

In [11]:
# sub_less_half(n), nice(n), lucas(n)

def subs_less_half(n):
    M = n//2 + 1
    return sum(math.comb(n, r) for r in range(0, M))

def nice(n):
    return numerical_approx(n, digits=4)

def lucas(n):
    phi = (1+sqrt(5))/2
    return round(phi^n + (1-phi)^n)

In [3]:
# tuple_orbits(G, k), num_forb_subs(G, gs, hs), helper functions

# Returns the orbits of G^k under the action of Aut(G)
def tuple_orbits(G, k):
    elements = G.Elements()
    aut_group = libgap.AutomorphismGroup(G)
    aut_order = aut_group.Order()
    tuple_orbits = []
    
    def build_orbits(current_stab, current_tuple):
        if len(current_tuple) == k:
            # Orbit-Stabilizer Theorem: |Orbit| = |G| / |Stabilizer|
            orbit_size = int(aut_order / current_stab.Order())
            tuple_orbits.append((current_tuple, orbit_size))
            return
        
        # Find the orbits of the current stabilizer acting on the group elements
        orbits = libgap.Orbits(current_stab, elements)
        
        for orb in orbits:
            rep = orb[0]
            
            # The new stabilizer is the subgroup of current_stab that also fixes 'rep'
            next_stab = libgap.Stabilizer(current_stab, rep)
            
            # Recurse deeper, adding this representative to our tuple
            build_orbits(next_stab, current_tuple + (rep,))
            
    build_orbits(aut_group, ())
    return tuple_orbits

class GroupContext:
    def __init__(self, G):
        """Precomputes the group structures for O(1) lookups."""
        # Convert G to libgap and extract elements
        self.G = libgap(G)
        self.elements = list(self.G.Elements())
        self.n = len(self.elements)
        
        # O(1) dictionary mapping a libgap element to its integer index
        self.elem_to_idx = {elem: i for i, elem in enumerate(self.elements)}
        
        # Precompute the Cayley Table (M) and Inverse Table (Inv)
        # M[i][j] gives the index of elements[i] * elements[j]
        self.M = [[0] * self.n for _ in range(self.n)]
        self.Inv = [0] * self.n
        
        for i, x in enumerate(self.elements):
            self.Inv[i] = self.elem_to_idx[x.Inverse()]
            for j, y in enumerate(self.elements):
                self.M[i][j] = self.elem_to_idx[x * y]
                
        # Precompute Fibonacci numbers (for path graphs)
        self.Fib = [0, 1, 1]
        for _ in range(self.n + 2): 
            self.Fib.append(self.Fib[-1] + self.Fib[-2])
            
        # Precompute Lucas numbers (for cycle graphs)
        self.Luc = [2, 1, 3] # L_0=2, L_1=1, L_2=3
        for _ in range(self.n): 
            self.Luc.append(self.Luc[-1] + self.Luc[-2])


# Counds the number of independent sets in a forbiddance graph whose structure
# is determined by S_conds
def count_independent_sets_forbiddance_gh(ctx, S_conds):
    n = ctx.n
    adj = [set() for _ in range(n)]
    forbidden = set()

    # 1. Build the forbiddance graph using O(1) Cayley table lookups
    for cond_type, val in S_conds:
        val_idx = ctx.elem_to_idx[val]
        
        if cond_type == 'g':
            for i in range(n):
                # Right-neighbor: y = x^-1 * val
                j = ctx.M[ctx.Inv[i]][val_idx]
                if i == j: forbidden.add(i)
                else:
                    adj[i].add(j)
                    adj[j].add(i)
                    
                # Left-neighbor: y2 = val * x^-1
                j2 = ctx.M[val_idx][ctx.Inv[i]]
                if i == j2: forbidden.add(i)
                else:
                    adj[i].add(j2)
                    adj[j2].add(i)
                    
        elif cond_type == 'h':
            for i in range(n):
                # Role of x: y = h^-1 * x
                j = ctx.M[ctx.Inv[val_idx]][i]
                if i == j: forbidden.add(i)
                else:
                    adj[i].add(j)
                    adj[j].add(i)
                    
                # Role of y: x = h * y
                j2 = ctx.M[val_idx][i]
                if i == j2: forbidden.add(i)
                else:
                    adj[i].add(j2)
                    adj[j2].add(i)

    memo = {}

    # 2. Branch-and-reduce solver
    def solve(active_nodes):
        k = len(active_nodes)
        if k == 0: return 1

        # A. Quick DFS to extract exactly ONE connected component
        start_node = next(iter(active_nodes))
        comp = {start_node}
        stack = [start_node]
        
        while stack:
            curr = stack.pop()
            for nbr in adj[curr]:
                if nbr in active_nodes and nbr not in comp:
                    comp.add(nbr)
                    stack.append(nbr)

        # B. Divide and Conquer
        if len(comp) < k:
            return solve(frozenset(comp)) * solve(active_nodes - comp)

        # C. Memoization
        if active_nodes in memo:
            return memo[active_nodes]

        # D. Pivot Selection & Topological Short-Circuit
        deg_counts = {u: len(adj[u] & active_nodes) for u in active_nodes}
        max_deg = max(deg_counts.values())
        
        if max_deg <= 2:
            min_deg = min(deg_counts.values())
            if min_deg == 2:
                res = ctx.Luc[k]         # It's a cycle graph
            else:
                res = ctx.Fib[k + 2]     # It's a path graph
            
            memo[active_nodes] = res
            return res
            
        # E. Branch on highest degree vertex
        v = next(u for u, d in deg_counts.items() if d == max_deg)
        
        res_exclude = solve(active_nodes - {v})
        neighbors = adj[v] & active_nodes
        res_include = solve(active_nodes - {v} - neighbors)
        
        res = res_exclude + res_include
        memo[active_nodes] = res
        return res

    # 3. Start solver stripped of forbidden nodes
    initial_nodes = frozenset(set(range(n)) - forbidden)
    return solve(initial_nodes)

# Described at the top of the file
def num_forb_subs(G, gs, hs):
    ctx = GroupContext(G)
    # Dictionary from keys preserves insertion order while stripping duplicates
    unique_g = list(dict.fromkeys(gs))
    unique_h = list(dict.fromkeys(hs))
    
    conditions = [('g', g) for g in unique_g] + [('h', h) for h in unique_h]
    k_total = len(conditions)
    
    total_not_all = 0
    
    for m in range(1, k_total + 1):
        sign = (-1) ** (m - 1)
        for S_conds in combinations(conditions, m):
            i_FS = count_independent_sets_forbiddance_gh(ctx, S_conds)
            total_not_all += sign * i_FS
            
    return total_not_all

In [47]:
# compute_mom(G,r), bal_bound(G), var_bound(G)

# Compute the r'th moment of d(A)
def compute_mom(G, r, verbatim=True):
    elms = G.Elements()
    n = len(elms)
    orbits = tuple_orbits(G, r)

    if verbatim:
        orbs = tqdm(orbits)
    else:
        orbs = orbits
    
    tot = 0
    for rep, orbit_size in orbs:
        val = 0
        for m in range(0, r+1):
            sgn = (-1)**(m+1)
            binom = binomial(r,m)

            split = r-m
            forbs = num_forb_subs(G, rep[:split], rep[split:])

            val += sgn*binom*forbs
            
        tot += orbit_size*val

    return tot

# Compute an upper bound for 2^n - B
def bal_bound(G, verbatim=True):
    elms = G.Elements()
    n = len(elms)
    orbits = tuple_orbits(G, 1)
    if verbatim:
        orbs = tqdm(orbits)
    else:
        orbs = orbits

    tot = 0
    for rep, orbit_size in orbs:
        tot += orbit_size * num_forb_subs(G, rep, rep)
    return tot

# Compute an upper bound for the second moment
def var_bound(G, verbatim=True):
    elms = G.Elements()
    n = len(elms)
    orbits = tuple_orbits(G, 1)
    if verbatim:
        orbs = tqdm(orbits)
    else:
        orbs = orbits

    tot = n * (n-1) * 19^(n/6)
    for rep, orbit_size in orbs:
        tot += orbit_size * (num_forb_subs(G, rep, []) + num_forb_subs(G, [], rep))
    return tot

Below is $M_1 =$ `mom1`, $M_2 =$ `mom2`, and $2^n - B =$ `bals`.

The goal is to have
$$ \frac{M_1^2}{M_2(2^n-B)} > \frac 12 $$

In [9]:
# List of number of balanced subsets for dihedral groups for k=0 to 30

dihedral_bals = [1, 4, 16, 46, 184, 684, 2830, 11008, 46584, 192250, 
                 806396, 3328472, 13788886, 56881972, 233244664, 957162366, 3897446296, 15877984160, 
                 64309943398, 260721882046, 1052085090244, 4247491679638, 17086382476476, 68788510296734,
                 276258844902918, 1109610722447684, 4449554721586276, 17848857181226662, 71515396060677424,
                 286563614409785468, 1147452736497479150]

In [46]:
"""
Compute the fraction with
- True first moment
- True second moment
- True balanced
"""
for k in range(3, 31):
    G = libgap(DihedralGroup(k))
    elms = G.Elements()
    n = len(elms)
    
    mom1 = compute_mom(G,1,verbatim=False)
    mom2 = compute_mom(G,2,verbatim=False)
    bals = 2**n - dihedral_bals[k]
    
    frc = (mom1**2) / (mom2 * bals)
    if (frc > 1/2):
        print(f"Success with k={k}: the ratio is ~{nice(frc)}")
    else:
        print(f"Failure with k={k}: the ratio is ~{nice(frc)}")
    

Success with k=3: the ratio is ~0.8889
Success with k=4: the ratio is ~0.6154
Success with k=5: the ratio is ~0.7686
Success with k=6: the ratio is ~0.5778
Success with k=7: the ratio is ~0.6377
Success with k=8: the ratio is ~0.5517
Success with k=9: the ratio is ~0.6041
Success with k=10: the ratio is ~0.5161
Success with k=11: the ratio is ~0.5755
Success with k=12: the ratio is ~0.5200
Success with k=13: the ratio is ~0.5661
Success with k=14: the ratio is ~0.5006
Success with k=15: the ratio is ~0.5745
Success with k=16: the ratio is ~0.5297
Success with k=17: the ratio is ~0.5721
Success with k=18: the ratio is ~0.5183
Success with k=19: the ratio is ~0.5843
Success with k=20: the ratio is ~0.5525
Success with k=21: the ratio is ~0.6073
Success with k=22: the ratio is ~0.5460
Success with k=23: the ratio is ~0.6210
Success with k=24: the ratio is ~0.5971
Success with k=25: the ratio is ~0.6452
Success with k=26: the ratio is ~0.5879
Success with k=27: the ratio is ~0.6732
Success

In [42]:
"""
Compute the fraction with
- True first moment
- True second moment
- Upper bound balanced
This takes the longest of the three
"""
for k in range(31, 47):
    G = libgap(DihedralGroup(k))
    elms = G.Elements()
    n = len(elms)
    
    mom1 = compute_mom(G,1,verbatim=False)
    mom2 = compute_mom(G,2,verbatim=True)
    bals = bal_bound(G,verbatim=False)
    
    frc = (mom1**2) / (mom2 * bals)
    if (frc > 1/2):
        print(f"Success with k={k}: the desired ratio is ~{nice(frc)}")
    else:
        print(f"Failure with k={k}: the desired ratio is ~{nice(frc)}")

ERROR! Session/line number was not unique in database. History logging moved to new session 244


  0%|          | 0/39 [00:00<?, ?it/s]

Success with k=31: the desired ratio is ~0.5780


  0%|          | 0/112 [00:00<?, ?it/s]

Success with k=32: the desired ratio is ~0.5603


...

Success with k=44: the desired ratio is ~0.7806


  0%|          | 0/137 [00:00<?, ?it/s]

Success with k=45: the desired ratio is ~0.8335


  0%|          | 0/112 [00:00<?, ?it/s]

Success with k=46: the desired ratio is ~0.7696


In [45]:
"""
Compute the fraction with
- True first moment
- Upper bound second moment
- Upper bound balanced
"""
for k in range(47, 55):
    G = libgap(DihedralGroup(k))
    elms = G.Elements()
    n = len(elms)

    mom1 = compute_mom(G,1,verbatim=False)
    mom2 = var_bound(G,verbatim=False)
    bals = bal_bound(G,verbatim=False)

    frc = (mom1**2) / (mom2 * bals)
    if (frc > 1/2):
        print(f"Success with k={k}: the desired ratio is ~{nice(frc)}")
    else:
        print(f"Failure with k={k}: the desired ratio is ~{nice(frc)}")
    

Success with k=47: the desired ratio is ~0.5207
Success with k=48: the desired ratio is ~0.5309
Success with k=49: the desired ratio is ~0.5693
Success with k=50: the desired ratio is ~0.5520
Success with k=51: the desired ratio is ~0.6175
Success with k=52: the desired ratio is ~0.6224
Success with k=53: the desired ratio is ~0.6616
Success with k=54: the desired ratio is ~0.6381
